# End-to-End BYOC Image Filtering for Vertex AI Search

This notebook demonstrates how to stage PDFs, extract and classify images using Document AI and Gemini to identify charts and graphs, and ingest the processed content into Vertex AI Search using Bring Your Own Chunks (BYOC).

### Step 0: Install Dependencies & Authenticate

In [ ]:
!pip install google-cloud-storage google-cloud-documentai google-cloud-discoveryengine vertexai pymupdf google-auth -q

In [ ]:
import os
import time
import json
import base64
import fitz  # PyMuPDF
from google.cloud import storage
from google.cloud import documentai
from google.cloud import discoveryengine_v1alpha as discoveryengine
from google.api_core.client_options import ClientOptions
import vertexai
from vertexai.generative_models import GenerativeModel, Part
import google.auth

# --- CONFIGURATION ---
# PLEASE REPLACE THIS WITH YOUR PROJECT ID
PROJECT_ID = "YOUR_PROJECT_ID_HERE"
LOCATION = "global" # For Vertex Search
DOCAI_LOCATION = "us" # For Document AI

TIMESTAMP = int(time.time())
BUCKET_NAME = f"{PROJECT_ID}-byoc-staging-{TIMESTAMP}"
STAGING_DIR = "./staging"
JSONL_OUTPUT_FILE = "byoc_chunks.jsonl"

DATA_STORE_ID = f"byoc-filtered-ds-{TIMESTAMP}"
ENGINE_ID = f"byoc-filtered-app-{TIMESTAMP}"
PROCESSOR_DISPLAY_NAME = f"byoc-layout-parser-{TIMESTAMP}"

print(f"Project ID: {PROJECT_ID}")
print(f"Bucket: {BUCKET_NAME}")
print(f"Output JSONL: {JSONL_OUTPUT_FILE}")


### Step 1: Create Staging Directory, GCS Bucket, and Upload PDFs

In [ ]:
def setup_staging_and_upload():
    if not os.path.exists(STAGING_DIR):
        os.makedirs(STAGING_DIR)
        print(f"Created staging directory at {STAGING_DIR}. Please place your PDFs here before proceeding.")
        return False

    pdf_files = [f for f in os.listdir(STAGING_DIR) if f.endswith('.pdf')]
    if not pdf_files:
        print(f"No PDFs found in {STAGING_DIR}. Please add PDFs before continuing.")
        return False

    storage_client = storage.Client(project=PROJECT_ID)
    try:
        bucket = storage_client.get_bucket(BUCKET_NAME)
        print(f"Bucket {BUCKET_NAME} already exists.")
    except Exception:
        bucket = storage_client.create_bucket(BUCKET_NAME, project=PROJECT_ID, location="US")
        print(f"Created bucket {BUCKET_NAME}")

    for filename in pdf_files:
        local_path = os.path.join(STAGING_DIR, filename)
        blob = bucket.blob(f"pdfs/{filename}")
        blob.upload_from_filename(local_path)
        print(f"Uploaded {filename} to gs://{BUCKET_NAME}/pdfs/{filename}")
    return True

ready_to_proceed = setup_staging_and_upload()


### Step 2: Initialize Document AI & Create Layout Parser
We use a Layout Parser to detect where all images and blocks are on the page.

In [ ]:
def create_layout_processor():
    opts = ClientOptions(api_endpoint=f"{DOCAI_LOCATION}-documentai.googleapis.com")
    docai_client = documentai.DocumentProcessorServiceClient(client_options=opts)
    parent = docai_client.common_location_path(PROJECT_ID, DOCAI_LOCATION)

    print(f"Creating Document AI Layout processor: {PROCESSOR_DISPLAY_NAME}")
    try:
        processor = docai_client.create_processor(
            parent=parent,
            processor=documentai.Processor(
                display_name=PROCESSOR_DISPLAY_NAME,
                type_="LAYOUT_PARSER_PROCESSOR"
            ),
        )
        print(f"Processor name: {processor.name}")
        return processor.name
    except Exception as e:
        print(f"Error creating processor: {e}")
        return None

if ready_to_proceed:
    PROCESSOR_NAME = create_layout_processor()
else:
    PROCESSOR_NAME = None


### Step 3: PyMuPDF Extraction & Gemini Classification Pipeline
We extract the detected boundaries, classify them with Gemini, and capture metadata for charts/graphs.

In [ ]:
if PROCESSOR_NAME:
    vertexai.init(project=PROJECT_ID, location="us-central1")
    gemini_model = GenerativeModel("gemini-1.5-flash-001")

def classify_image_with_gemini(image_bytes):
    """Returns 'chart', 'graph', 'photo', or 'other'."""
    prompt = """
    Analyze the provided image and classify it into exactly one of these categories: 
    'chart', 'graph', 'photo', or 'other'. 
    Respond with ONLY the category name in lowercase.
    """
    image_part = Part.from_data(data=image_bytes, mime_type="image/png")
    try:
        response = gemini_model.generate_content([prompt, image_part])
        category = response.text.strip().lower()
        return category if category in ['chart', 'graph', 'photo', 'other'] else 'other'
    except Exception as e:
        print(f"Gemini Error: {e}")
        return 'other'

def extract_images_from_pdf(local_pdf_path):
    """
    1. Processes document via Document AI Layout Parser.
    2. Extracts images using PyMuPDF bounding boxes.
    3. Classifies them.
    4. Returns extracted chart/graph chunks and remaining text chunks.
    """
    opts = ClientOptions(api_endpoint=f"{DOCAI_LOCATION}-documentai.googleapis.com")
    docai_client = documentai.DocumentProcessorServiceClient(client_options=opts)
    with open(local_pdf_path, "rb") as f:
        raw_doc = documentai.RawDocument(content=f.read(), mime_type="application/pdf")
        
    request = documentai.ProcessRequest(name=PROCESSOR_NAME, raw_document=raw_doc)
    docai_result = docai_client.process_document(request=request)
    document = docai_result.document

    pdf_doc = fitz.open(local_pdf_path)
    
    custom_chunks = []
    image_counter = 0

    for page_num, page in enumerate(document.pages):
        pdf_page = pdf_doc[page_num]
        page_width = pdf_page.rect.width
        page_height = pdf_page.rect.height
        
        for block_idx, image_block in enumerate(page.image_block):
            if not image_block.layout.bounding_poly:
                continue
            vertices = image_block.layout.bounding_poly.normalized_vertices
            x_coords = [v.x for v in vertices]
            y_coords = [v.y for v in vertices]
            
            rect = fitz.Rect(
                min(x_coords) * page_width,
                min(y_coords) * page_height,
                max(x_coords) * page_width,
                max(y_coords) * page_height
            )
            
            pix = pdf_page.get_pixmap(clip=rect)
            image_bytes = pix.tobytes("png")
            
            category = classify_image_with_gemini(image_bytes)
            print(f"Page {page_num+1}, Detected Image {image_counter}: classified as '{category}'")
            
            if category in ['chart', 'graph']:
                b64_image = base64.b64encode(image_bytes).decode('utf-8')
                print(f" -> Extracting metadata for {category}.")
                # Redact from PDF to remove it visually
                pdf_page.add_redact_annot(rect, fill=(1,1,1))
                pdf_page.apply_redactions()
                
                chunk = {
                    "chunkId": f"{os.path.basename(local_pdf_path)}_page{page_num+1}_img{image_counter}",
                    "content": f"Extracted {category} from {os.path.basename(local_pdf_path)} page {page_num+1}.", 
                    "structData": {
                        "type": category,
                        "source_file": os.path.basename(local_pdf_path),
                        "page": page_num + 1,
                        "is_chart_graph_extract": True
                    },
                    "chunkFields": [
                        {
                            "name": "visual_asset",
                            "imageChunkField": {
                                "blobAssetId": f"blob_{os.path.basename(local_pdf_path)}_{image_counter}"
                            }
                        }
                    ],
                    "_raw_b64": b64_image, 
                    "_blob_id": f"blob_{os.path.basename(local_pdf_path)}_{image_counter}"
                }
                custom_chunks.append(chunk)
                
            image_counter += 1

    cleaned_path = local_pdf_path.replace(".pdf", "_cleaned.pdf")
    pdf_doc.save(cleaned_path)
    pdf_doc.close()
    
    return document.text, custom_chunks


### Step 4: Process All Staged PDFs & Generate JSONL
Run the pipeline on our PDFs and output to JSONL for Vertex AI Search BYOC.

In [ ]:
all_jsonl_records = []

if PROCESSOR_NAME and ready_to_proceed:
    pdf_files = [f for f in os.listdir(STAGING_DIR) if f.endswith('.pdf') and not f.endswith('_cleaned.pdf')]
    
    for filename in pdf_files:
        print(f"\nProcessing {filename}...")
        local_path = os.path.join(STAGING_DIR, filename)
        
        full_text, custom_chunks = extract_images_from_pdf(local_path)
        
        blob_assets = []
        chunks_for_doc = []
        
        for chunk in custom_chunks:
            blob_assets.append({
                "name": chunk["_blob_id"],
                "content": chunk["_raw_b64"],
                "mimeType": "image/png"
            })
            del chunk["_raw_b64"]
            del chunk["_blob_id"]
            chunks_for_doc.append(chunk)

        if full_text:
             chunks_for_doc.append({
                 "chunkId": f"{filename}_main_text",
                 "content": full_text[:8000],  # Text size might need to be chunked further in production
                 "structData": { "source_file": filename }
             })

        document_payload = {
             "id": filename.replace(".pdf", "").replace(".", "_").replace(" ", "_"),
             "structData": {"file_name": filename},
             "jsonData": json.dumps({
                 "blobAssets": blob_assets,
                 "chunkedDocument": {
                     "chunks": chunks_for_doc
                 }
             }),
             "content": {
                 "mimeType": "text/plain",
                 "rawBytes": "RGVtbw=="
             }
        }
        all_jsonl_records.append(document_payload)
        
    with open(JSONL_OUTPUT_FILE, 'w') as f:
        for record in all_jsonl_records:
            f.write(json.dumps(record) + '\n')
            
    print(f"\nWrote {len(all_jsonl_records)} documents to {JSONL_OUTPUT_FILE}")
    
    storage_client = storage.Client(project=PROJECT_ID)
    bucket = storage_client.get_bucket(BUCKET_NAME)
    blob = bucket.blob(JSONL_OUTPUT_FILE)
    blob.upload_from_filename(JSONL_OUTPUT_FILE)
    print(f"Uploaded JSONL to gs://{BUCKET_NAME}/{JSONL_OUTPUT_FILE}")
else:
    print("Prerequisites not met. Skipping generation.")

### Step 5: Vertex AI Search - Create App & Ingest Data
Create an Unstructured Data Store and a Search Engine to receive the BYOC JSONL.

In [ ]:
def create_data_store():
    client_options = (
        ClientOptions(api_endpoint=f"{LOCATION}-discoveryengine.googleapis.com")
        if LOCATION != "global"
        else None
    )
    ds_client = discoveryengine.DataStoreServiceClient(client_options=client_options)
    parent = f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection"

    data_store = discoveryengine.DataStore(
        display_name="BYOC Filtered Demo Store",
        industry_vertical=discoveryengine.IndustryVertical.GENERIC,
        solution_types=[discoveryengine.SolutionType.SOLUTION_TYPE_SEARCH],
        content_config=discoveryengine.DataStore.ContentConfig.CONTENT_REQUIRED,
    )
    print(f"Creating Data Store: {DATA_STORE_ID}...")
    try:
        operation = ds_client.create_data_store(
            request=discoveryengine.CreateDataStoreRequest(
                parent=parent,
                data_store_id=DATA_STORE_ID,
                data_store=data_store,
            )
        )
        return operation.result().name
    except Exception as e:
        print(f"Store exists or error: {e}")
        return f"{parent}/dataStores/{DATA_STORE_ID}"

def create_engine():
    client_options = (
        ClientOptions(api_endpoint=f"{LOCATION}-discoveryengine.googleapis.com")
        if LOCATION != "global"
        else None
    )
    engine_client = discoveryengine.EngineServiceClient(client_options=client_options)
    parent = f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection"

    engine = discoveryengine.Engine(
        display_name="BYOC Filtered Demo App",
        solution_type=discoveryengine.SolutionType.SOLUTION_TYPE_SEARCH,
        data_store_ids=[DATA_STORE_ID],
        search_engine_config=discoveryengine.Engine.SearchEngineConfig(
            search_tier=discoveryengine.SearchTier.SEARCH_TIER_ENTERPRISE,
            search_add_ons=[discoveryengine.SearchAddOn.SEARCH_ADD_ON_LLM],
        ),
    )
    print(f"Creating Engine: {ENGINE_ID}...")
    try:
        operation = engine_client.create_engine(
            request=discoveryengine.CreateEngineRequest(
                parent=parent,
                engine_id=ENGINE_ID,
                engine=engine,
            )
        )
        return operation.result().name
    except Exception as e:
        print(f"Engine exists or error: {e}")
        return f"{parent}/engines/{ENGINE_ID}"

if PROCESSOR_NAME and ready_to_proceed:
    ds_name = create_data_store()
    engine_name = create_engine()
    print("Waiting 15 seconds for backend to sync...")
    time.sleep(15)


### Step 6: Import Documents from JSONL

In [ ]:
import requests

def import_jsonl_via_rest():
    creds, project = google.auth.default()
    auth_req = google.auth.transport.requests.Request()
    creds.refresh(auth_req)
    access_token = creds.token

    base_url = f"https://{LOCATION}-discoveryengine.googleapis.com" if LOCATION != "global" else "https://discoveryengine.googleapis.com"
    url = f"{base_url}/v1alpha/projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection/dataStores/{DATA_STORE_ID}/branches/default_branch/documents:import"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }
    
    payload = {
      "gcsSource": {
        "inputUris": [
          f"gs://{BUCKET_NAME}/{JSONL_OUTPUT_FILE}"
        ],
        "dataSchema": "custom"
      },
      "reconciliationMode": "INCREMENTAL"
    }

    print(f"Triggering Import from GCS: gs://{BUCKET_NAME}/{JSONL_OUTPUT_FILE}")
    response = requests.post(url, headers=headers, json=payload)

    if response.status_code == 200:
        print("\n[SUCCESS] Import operation initiated successfully!")
        print("Response:", json.dumps(response.json(), indent=2))
        print("\nCheck your Cloud Console to view progress.")
    else:
        print(f"\n[ERROR {response.status_code}]: {response.text}")

if PROCESSOR_NAME and ready_to_proceed:
    import_jsonl_via_rest()


### Step 7: Query the App using Answer API

We can now query our Vertex AI Search App to retrieve answers based on the BYOC chunks. Because we ingested images alongside chunks and used `blobAssets`, the response should return `blobAttachments` allowing us to visually display charts or graphs that were filtered and mapped.

In [ ]:
def query_answer_rest():
    """Queries the App using Answer API."""
    creds, project = google.auth.default()
    auth_req = google.auth.transport.requests.Request()
    creds.refresh(auth_req)
    access_token = creds.token

    base_url = f"https://{LOCATION}-discoveryengine.googleapis.com" if LOCATION != "global" else "https://discoveryengine.googleapis.com"
    url = f"{base_url}/v1alpha/projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection/engines/{ENGINE_ID}/servingConfigs/default_serving_config:answer"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    query_text = "What is the main topic of the uploaded PDFs?" # Feel free to change

    payload = {
        "query": { "text": query_text },
        "answerGenerationSpec": {
            "includeCitations": True,
            "multimodalSpec": {"imageSource": "CORPUS_IMAGE_ONLY"},
            "modelSpec": { "modelVersion": "stable" }
        }
    }

    print(f"\nSending query: '{query_text}'...")
    response = requests.post(url, headers=headers, json=payload)

    if response.status_code == 200:
        data = response.json()
        answer = data.get("answer", {})
        citations = answer.get("citations", [])
        
        print("\n--- Answer ---")
        print(answer.get("answerText", "No text returned."))
        
        for citation in citations:
            for source in citation.get("sources", []):
                chunk_info = source.get("chunkInfo", {})
                print("\n--- Citation Source Found ---")
                
                blob_attachments = chunk_info.get("blobAttachments", [])
                if blob_attachments:
                    print(f"[SUCCESS] Found {len(blob_attachments)} Blob Attachment(s)!")
                    for blob in blob_attachments:
                        print(f" -> Blob Name: {blob.get('name')}")
                else:
                    print("[INFO] No Blob Attachments for this source.")
    else:
        print(f"\n[ERROR {response.status_code}]: {response.text}")

if PROCESSOR_NAME and ready_to_proceed:
    # wait for parsing and import to be fully indexed before query 
    # (In a real scenario, this might take 5-10 minutes)
    query_answer_rest()
